## STEP 0 
Import Libraries, Connect to Snowflake, & Initialize NLP Pipelines

In [1]:
import os
import snowflake.connector
from snowflake.connector.pandas_tools import write_pandas
import torch
from transformers import pipeline

# this basically means "smoke em if you got em" where the "em" is NVIDIA GPU
DEVICE = 0 if torch.cuda.is_available() else -1

SF_USR = os.getenv('SF_USR')
SF_ID  = os.getenv('SF_ID')
SF_WH  = os.getenv('SF_WH')
SF_DB  = os.getenv('SF_DB')
SF_SC  = os.getenv('SF_SC')
SF_RL  = os.getenv('SF_RL')

# connect to database and init a cursor for querying
xct_params = {
    "user":                 os.getenv('SF_USR')
   ,"account":              os.getenv('SF_ID')
   ,"warehouse":            os.getenv('SF_WH')
   ,"database":             os.getenv('SF_DB')
   ,"schema":               os.getenv('SF_SC')
   ,"role":                 os.getenv('SF_RL')
   ,"private_key_file":     os.getenv('PRIVATE_KEY_PATH')
   ,"private_key_file_pwd": os.getenv('PRIVATE_KEY_PASSPHRASE')
   ,"authenticator":        os.getenv('SF_AUTH')
}
SF_XCT = snowflake.connector.connect(**xct_params)
CSR = SF_XCT.cursor()

# sentiment analyzer doo-dad instantiation
PIPL_SENT = pipeline(
    "sentiment-analysis",
    model="cardiffnlp/twitter-roberta-base-sentiment",
    tokenizer="cardiffnlp/twitter-roberta-base-sentiment",
    device=DEVICE,
    truncation=True,
    max_length = 512 
)
## this sentiment model has the below mapping that indicates the overall sentiment returned
## for verification, see this link:
##      https://huggingface.co/cardiffnlp/twitter-roberta-base-sentiment

# named-entity recognition doo-dad instantiation
PIPL_NER = pipeline(
    "ner",
    model="dslim/bert-base-NER",
    tokenizer="dslim/bert-base-NER",
    aggregation_strategy="simple",
    device=DEVICE,
    batch_size=256 
)

Device set to use cuda:0
Some weights of the model checkpoint at dslim/bert-base-NER were not used when initializing BertForTokenClassification: ['bert.pooler.dense.bias', 'bert.pooler.dense.weight']
- This IS expected if you are initializing BertForTokenClassification from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing BertForTokenClassification from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).
Device set to use cuda:0


## STEP 1
Ingest Post-Text into Memory

In [ ]:
import pandas as pd
query = f"""
select content_id
      ,usa_timestamp as post_created_usa_timestamp
      ,post_text
from {SF_DB}.{SF_SC}.firehose_processed
where (first_detected_language = 'English'
   or  first_detected_language is null
   )
  and post_created_usa_timestamp <= to_timestamp_tz('2025-05-24 23:59:59+0000')
;
"""
# and post_created_usa_timestamp >= (select nvl(max(post_created_usa_timestamp), '1900-00-00 00:00:00+0000)
#                                    from {SF_DB}.{SF_SC}.firehose_nlp_labeled) 
#
CSR.execute(query)
DATA = CSR.fetch_pandas_all()

,CONTENT_ID,POST_CREATED_USA_TIMESTAMP_ORIG,POST_TEXT,SENTIMENT_ANALYSIS,NER_ANALYSIS
0,bafyreidlhptytpfax76fz6mjabkpo577mksdzvdvctcdl...,2025-05-24 03:04:35.911978+00:00,Line 1 Yonge-University: There is no subway se...,"{'label': 'LABEL_1', 'score': 0.5646804571151733}","[{'entity_group': 'ORG', 'score': 0.68038857, ..."
1,bafyreiakmq7t4kffnuvrmogxidjqzjoqq3cq3pmfg5wns...,2025-05-24 07:18:48.690129+00:00,🇳🇴 Supreme Court: Lawyer from firm with assign...,"{'label': 'LABEL_1', 'score': 0.8142656087875366}",[]
2,bafyreiboezd6hrlmvozajlysdsbbdi2ejeojrswqn7ttm...,2025-05-24 16:19:16.220606+00:00,Overzicht Nacompetitieschema’s (West II)\nbron...,"{'label': 'LABEL_1', 'score': 0.8204275965690613}","[{'entity_group': 'ORG', 'score': 0.99271286, ..."
3,bafyreih5cgmv2qro32bem65me4o7xq5lkwcgjkfmli4od...,2025-05-24 01:21:44.718699+00:00,#XREALOne で エンジェルフライト4話みながら #踏み台昇降 60分\n\n今回は平...,"{'label': 'LABEL_1', 'score': 0.7604115009307861}",[]
4,bafyreigw6ggme22yw7dtkfq7igpia752vycw5fixfgstb...,2025-05-24 16:21:45.995908+00:00,電気、通信、道路に投資して収益を得られる？？!#ai #fx #投資 #経済 #教育 #資産...,"{'label': 'LABEL_1', 'score': 0.6640048027038574}",[]
...,...,...,...,...,...
126310,bafyreibkzom6iwwikuecdpq63fyeousfcouvamsj337xb...,2025-05-23 19:20:29.707058+00:00,The man is a legend. Give him all the crap you...,"{'label': 'LABEL_2', 'score': 0.7466409802436829}","[{'entity_group': 'MISC', 'score': 0.9808088, ..."
126311,bafyreidy3xr2uvugod2y4lv4m6vol36fzycwldvda4vcb...,2025-05-23 22:21:20.196572+00:00,It became one of my favourites.,"{'label': 'LABEL_2', 'score': 0.9500476121902466}",[]
126312,bafyreiegf3f3ft4cxhxt3py5th2ib5q7bcvi6uwyrq63b...,2025-05-23 22:21:51.201279+00:00,Thank You,"{'label': 'LABEL_2', 'score': 0.8064097166061401}",[]
126313,bafyreiax47hxkmizacpmpx5a7cxdtfjk672x57pkgl53b...,2025-05-23 16:18:14.522528+00:00,Most of the university sector is now in that b...,"{'label': 'LABEL_0', 'score': 0.8869750499725342}","[{'entity_group': 'PER', 'score': 0.99402773, ..."


In [21]:
DATA

,CONTENT_ID,POST_CREATED_USA_TIMESTAMP_ORIG,POST_TEXT,SENTIMENT_ANALYSIS,NER_ANALYSIS
0,bafyreidlhptytpfax76fz6mjabkpo577mksdzvdvctcdl...,2025-05-24 03:04:35.911978+00:00,Line 1 Yonge-University: There is no subway se...,"{'label': 'LABEL_1', 'score': 0.5646804571151733}","[{'entity_group': 'ORG', 'score': 0.68038857, ..."
1,bafyreiakmq7t4kffnuvrmogxidjqzjoqq3cq3pmfg5wns...,2025-05-24 07:18:48.690129+00:00,🇳🇴 Supreme Court: Lawyer from firm with assign...,"{'label': 'LABEL_1', 'score': 0.8142656087875366}",[]
2,bafyreiboezd6hrlmvozajlysdsbbdi2ejeojrswqn7ttm...,2025-05-24 16:19:16.220606+00:00,Overzicht Nacompetitieschema’s (West II)\nbron...,"{'label': 'LABEL_1', 'score': 0.8204275965690613}","[{'entity_group': 'ORG', 'score': 0.99271286, ..."
3,bafyreih5cgmv2qro32bem65me4o7xq5lkwcgjkfmli4od...,2025-05-24 01:21:44.718699+00:00,#XREALOne で エンジェルフライト4話みながら #踏み台昇降 60分\n\n今回は平...,"{'label': 'LABEL_1', 'score': 0.7604115009307861}",[]
4,bafyreigw6ggme22yw7dtkfq7igpia752vycw5fixfgstb...,2025-05-24 16:21:45.995908+00:00,電気、通信、道路に投資して収益を得られる？？!#ai #fx #投資 #経済 #教育 #資産...,"{'label': 'LABEL_1', 'score': 0.6640048027038574}",[]
...,...,...,...,...,...
126310,bafyreibkzom6iwwikuecdpq63fyeousfcouvamsj337xb...,2025-05-23 19:20:29.707058+00:00,The man is a legend. Give him all the crap you...,"{'label': 'LABEL_2', 'score': 0.7466409802436829}","[{'entity_group': 'MISC', 'score': 0.9808088, ..."
126311,bafyreidy3xr2uvugod2y4lv4m6vol36fzycwldvda4vcb...,2025-05-23 22:21:20.196572+00:00,It became one of my favourites.,"{'label': 'LABEL_2', 'score': 0.9500476121902466}",[]
126312,bafyreiegf3f3ft4cxhxt3py5th2ib5q7bcvi6uwyrq63b...,2025-05-23 22:21:51.201279+00:00,Thank You,"{'label': 'LABEL_2', 'score': 0.8064097166061401}",[]
126313,bafyreiax47hxkmizacpmpx5a7cxdtfjk672x57pkgl53b...,2025-05-23 16:18:14.522528+00:00,Most of the university sector is now in that b...,"{'label': 'LABEL_0', 'score': 0.8869750499725342}","[{'entity_group': 'PER', 'score': 0.99402773, ..."


## STEP 2

Apply NLP Pipelines

In [3]:
sentiment_output = PIPL_SENT(DATA['POST_TEXT'].tolist())
ner_output       = PIPL_NER(DATA['POST_TEXT'].tolist())

## STEP 3
Stash Raw NLP Output as intermediate data in a Snowflake table

In [ ]:
DATA['SENTIMENT_ANALYSIS'] = sentiment_output
DATA['NER_ANALYSIS']       = ner_output
# POST_TEXT is a chunky column whose values must also be duplicated during processing-- don't store it in two tables
DATA.drop('POST_TEXT', axis=1, inplace=True)  


KeyError: "['POST_TEXT'] not found in axis"

In [28]:

write_pandas(SF_XCT, DATA
            ,table_name='INT_FIREHOSE_NLP'
            ,database=SF_DB.replace('"', '').upper()
            ,schema=SF_SC.replace('"', '').upper()
           )

/tmp/ipykernel_171752/562656167.py:1: UserWarning: Dataframe contains a datetime with timezone column, but 'use_logical_type=None'. This can result in dateimes being incorrectly written to Snowflake. Consider setting 'use_logical_type = True'
  write_pandas(SF_XCT, DATA


(True,
 1,
 126315,
 [('hfpbqmjrlv/file0.txt',
   'LOADED',
   126315,
   126315,
   1,
   0,
   None,
   None,
   None,
   None)])

## STEP 4
 Retrieve stashed data, blow it out into many processed rows per-post, then insert. Clear the stash when you're done.

In [ ]:
query = f"""
insert into {SF_DB}.{SF_SC}.firehose_nlp_labeled
with src as (
select a.content_id
      ,a.post_created_usa_timestamp
      ,b.readable_label_name as sentiment_detected_label
      ,cast(sentiment_analysis:score as number(5,4)) as sentiment_confidence_score
      ,row_number() over (
       partition by content_id
       order     by post_created_usa_timestamp, trim(a2.value:word, '"')
       ) as post_entity_number
      ,trim(a2.value:entity_group, '"') as ner_detected_group
      ,trim(a2.value:word, '"') as ner_detected_entity
      ,cast(a2.value:score as number(5,4)) as ner_confidence_score
from {SF_DB}.{SF_SC}.int_firehose_nlp a
left join table(flatten(input => parse_json(a.ner_analysis))) a2
left join {SF_DB}.{SF_SC}.label_map_roberta_base_sentiment b
       on trim(a.sentiment_analysis:label, '"') = b.model_label_name
)

select sha2(nvl(to_char(content_id), 'NULL') 
         || '||' 
         || nvl(to_char(post_entity_number), 'NULL')
       ) as analysis_id
      ,*
from src 
order by content_id
        ,post_created_usa_timestamp
        ,ner_detected_entity
"""
CSR.execute(query)


In [31]:

query = f'truncate table {SF_DB}.{SF_SC}.int_firehose_nlp'
CSR.execute(query)